In [82]:
import pandas as pd
import numpy as np

df = pd.read_csv("data.csv")
df.shape

(569, 33)

In [83]:
X = df.drop(["id", "diagnosis"], axis=1)
y = df["diagnosis"]

print(X.shape, y.shape)

(569, 31) (569,)


In [84]:
y.replace("M", 1, inplace=True)
y.replace("B", 0, inplace=True)

0      1
1      1
2      1
3      1
4      1
      ..
564    1
565    1
566    1
567    1
568    0
Name: diagnosis, Length: 569, dtype: object

In [85]:
def calculate_entropy(y):
    if len(y) == 0:
        return 0
    p = y.sum() / len(y)
    if p == 0 or p == 1:
        return 0
    return -(p * np.log2(p) + (1 - p) * np.log2(1-p))

In [86]:
def information_gain(parent, left_y, right_y):
    parent_entropy = calculate_entropy(parent)

    weight_left = len(left_y) / len(parent)
    weight_right  = len(right_y) / len(parent)

    child_entropy = weight_left * calculate_entropy(left_y) + weight_right * calculate_entropy(right_y)

    return parent_entropy - child_entropy


In [87]:
def get_best_split(X, y):
    best_gain = 0
    best_feature = None
    best_threshold = None

    for col_name in X.columns:
        for threshold in X[col_name].unique():
            mask = X[col_name] <= threshold

            left_y = y[mask]
            right_y = y[~mask]

            if len(left_y) == 0 or len(right_y) == 0:
                continue

            gain = information_gain(y, left_y, right_y)

            if gain > best_gain:
                best_gain = gain
                best_feature = col_name
                best_threshold = threshold

    return best_feature, best_threshold, best_gain

In [88]:
def build_tree(X, y, current_depth=0, max_depth=3):
    if current_depth >= max_depth or calculate_entropy(y) == 0:
        return y.mode()[0] # leaf node and return the most common label in that group

    best_feature, best_threshold, best_gain = get_best_split(X, y)

    if best_gain == 0:
        return y.mode()[0]

    mask = X[best_feature] <= best_threshold
    left_X, left_y = X[mask], y[mask]
    right_X, right_y = X[~mask], y[~mask]

    left_branch = build_tree(left_X, left_y, current_depth=current_depth+1)
    right_branch = build_tree(right_X, right_y, current_depth=current_depth+1)

    return {
        "feature": best_feature,
        "threshold": best_threshold,
        "left": left_branch,
        "right": right_branch
    }

In [89]:
tree = build_tree(X, y)

In [90]:
print(tree)

{'feature': 'perimeter_worst', 'threshold': np.float64(105.9), 'left': {'feature': 'concave points_worst', 'threshold': np.float64(0.1342), 'left': {'feature': 'area_se', 'threshold': np.float64(48.84), 'left': 0, 'right': 0}, 'right': {'feature': 'texture_worst', 'threshold': np.float64(27.2), 'left': 0, 'right': 1}}, 'right': {'feature': 'perimeter_worst', 'threshold': np.float64(117.2), 'left': {'feature': 'smoothness_worst', 'threshold': np.float64(0.1354), 'left': 0, 'right': 1}, 'right': {'feature': 'fractal_dimension_se', 'threshold': np.float64(0.001519), 'left': 0, 'right': 1}}}


In [91]:
def predict(patient, tree):
    if type(tree) != dict:
        return tree
    if patient[tree["feature"]] <= tree["threshold"]:
        return predict(patient, tree["left"])
    else:
        return predict(patient, tree["right"])

In [92]:
predictions = []

for index, row in X.iterrows():
    predictions.append(predict(row, tree))

In [96]:
from sklearn.metrics import classification_report

print(classification_report(y.astype(int), np.asarray(predictions, dtype=int)))

              precision    recall  f1-score   support

           0       0.95      1.00      0.98       357
           1       0.99      0.92      0.96       212

    accuracy                           0.97       569
   macro avg       0.97      0.96      0.97       569
weighted avg       0.97      0.97      0.97       569

